# DDoS Auswertung

## Notes:

instead of polars, the polars-lts-cpu package is used, to run on the old CPU's of the mobi8

## Imports

In [19]:
from pathlib import Path
import polars as pl
import matplotlib.pyplot as plt
import pyarrow.parquet as pq

## Globale Parameter

In [20]:
SERVER_IP = "141.22.28.227"

### external IANA protocol list to convert numbers to names

In [21]:
# read IANA list to convert number to text
lf_pro_nb = (
    pl.scan_csv("data/external/protocol-numbers-1.csv", ignore_errors=True)
      .select("Decimal", "Keyword")
).rename({"Keyword": "ip.proto.name", "Decimal": "ip.proto.num"})
lf_pro_nb.head(10).collect()

ip.proto.num,ip.proto.name
i64,str
0,"""HOPOPT"""
1,"""ICMP"""
2,"""IGMP"""
3,"""GGP"""
4,"""IPv4"""
5,"""ST"""
6,"""TCP"""
7,"""CBT"""
8,"""EGP"""


In [22]:
# lf = (
#     lf.join(lf_pro_nb, left_on="ip.proto", right_on="ip.proto.num", how="left")
#     .rename({"ip.proto": "ip.proto.num"})
#     .pipe(lambda lf: lf.select(
#     (cols := lf.collect_schema().names(), cols.insert(5, cols.pop(cols.index("ip.proto.name"))), cols)[-1]
# ))
# )

# lf.head(10).collect()

## 1. Überblick über die Daten (raw)

In [23]:
lf = pl.scan_parquet("data/interim/ddos_1/batch_1_2000.parquet")
df_sample = lf.limit(100).collect().sample(10)
df_sample

frame.datetime,frame.len,frame.protocols,ip.src,ip.dst,ip.proto,ip.ttl,ipv6.src,ipv6.dst,ipv6.hlim,udp.srcport,udp.dstport,udp.length,tcp.srcport,tcp.dstport,tcp.len,dtls.record.content_type,dtls.record.version,dtls.handshake.type,coap.version,coap.type,coap.code,coap.mid,coap.token,coap.token_len,coap.opt.uri_path,coap.opt.uri_query,coap.opt.block_number,coap.opt.block_mflag,coap.opt.block_size,coap.opt.observe,coap.response_to,coap.response_in,coap.payload_length,source_file
datetime[μs],i64,str,str,str,str,str,str,str,str,str,str,i64,str,str,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
2025-08-13 15:51:49.574079,1514,"""eth:ethertype:ip:udp:snmp""","""27.124.68.194""","""141.22.28.227""","""17""","""53""",null,null,null,"""161""","""10324""",2480,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""data/raw/ddos_1/13082025_DDOS_…"
2025-08-13 15:51:49.573754,1514,"""eth:ethertype:ip:udp:cldap""","""83.221.179.204""","""141.22.28.227""","""17""","""118""",null,null,null,"""389""","""47861""",2944,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""data/raw/ddos_1/13082025_DDOS_…"
2025-08-13 15:51:49.573915,861,"""eth:ethertype:ip:data""","""113.160.234.49""","""141.22.28.227""","""17""","""49""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""data/raw/ddos_1/13082025_DDOS_…"
2025-08-13 15:51:49.574189,786,"""eth:ethertype:ip:udp:snmp""","""208.77.179.64""","""141.22.28.227""","""17""","""45""",null,null,null,"""161""","""3364""",2362,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""data/raw/ddos_1/13082025_DDOS_…"
2025-08-13 15:51:49.573601,1514,"""eth:ethertype:ip:data""","""200.41.174.22""","""141.22.28.227""","""17""","""50""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""data/raw/ddos_1/13082025_DDOS_…"
2025-08-13 15:51:49.574188,1514,"""eth:ethertype:ip:data""","""38.52.217.173""","""141.22.28.227""","""17""","""52""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""data/raw/ddos_1/13082025_DDOS_…"
2025-08-13 15:51:49.573859,1506,"""eth:ethertype:ip:data""","""116.106.197.147""","""141.22.28.227""","""17""","""45""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""data/raw/ddos_1/13082025_DDOS_…"
2025-08-13 15:51:49.574189,778,"""eth:ethertype:ip:data""","""208.77.179.64""","""141.22.28.227""","""17""","""45""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""data/raw/ddos_1/13082025_DDOS_…"
2025-08-13 15:51:49.574454,1506,"""eth:ethertype:ip:udp:dns""","""82.79.235.118""","""141.22.28.227""","""17""","""54""",null,null,null,"""53""","""24312""",3747,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""data/raw/ddos_1/13082025_DDOS_…"


In [24]:
double_ip = lf.filter(
    pl.col("ip.ttl").str.contains(",") | pl.col("ip.proto").str.contains(",")
).collect()
print(double_ip.shape)
print(double_ip)

double_ports = lf.filter(
    pl.col("udp.dstport").str.contains(",") | pl.col("udp.srcport").str.contains(",") | pl.col("tcp.dstport").str.contains(",") | pl.col("udp.srcport").str.contains(",")
).collect()
print(double_ports.shape)
print(double_ports)

(13312618, 35)
shape: (13_312_618, 35)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ frame.dat ┆ frame.len ┆ frame.pro ┆ ip.src    ┆ … ┆ coap.resp ┆ coap.resp ┆ coap.payl ┆ source_f │
│ etime     ┆ ---       ┆ tocols    ┆ ---       ┆   ┆ onse_to   ┆ onse_in   ┆ oad_lengt ┆ ile      │
│ ---       ┆ i64       ┆ ---       ┆ str       ┆   ┆ ---       ┆ ---       ┆ h         ┆ ---      │
│ datetime[ ┆           ┆ str       ┆           ┆   ┆ str       ┆ str       ┆ ---       ┆ str      │
│ μs]       ┆           ┆           ┆           ┆   ┆           ┆           ┆ str       ┆          │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
│ 2025-08-1 ┆ 70        ┆ eth:ether ┆ 186.9.151 ┆ … ┆ null      ┆ null      ┆ null      ┆ data/raw │
│ 3 15:51:4 ┆           ┆ type:ip:i ┆ .108,141. ┆   ┆           ┆           ┆           ┆ /ddos_1/ │
│ 9.574617  ┆           ┆ cmp:ip:ud ┆ 22.28.227 ┆   

### Datenübersicht

#### Incoming vs. Outgoing

In [25]:
def split_direction(lf: pl.LazyFrame, server_ip):
    lf_out = lf.filter(pl.col("ip.src") == server_ip)
    lf_in = lf.filter(pl.col("ip.dst") == server_ip)
    lf_null = lf.filter(
        (pl.col("ip.dst") != server_ip) & (pl.col("ip.src") != server_ip)
    )
    lf_null = lf.filter(
        pl.col("ip.src").is_null() | pl.col("ip.dst").is_null()
    )
    
    return lf_out, lf_in, lf_null

In [26]:
lf_out, lf_in, lf_null = split_direction(lf, SERVER_IP)

size = lf.select(pl.len()).collect().item()
size_out = lf_out.select(pl.len()).collect().item()
size_in = lf_in.select(pl.len()).collect().item()
size_null = lf_null.select(pl.len()).collect().item()

print(f"Packets gesamt:{size:3,}\n")
print(f"Anteil out: {size_out/size:.6}%")
print(f"Packets out:", size_out)
print(f"Anteil in: {size_in/size:.2}%")
print(f"Packets in:", size_in)
print(f"Anteil null: {size_null / size:.6f}%")
print(f"Packets null:", size_null)

Packets gesamt:1,392,529,795

Anteil out: 5.92662e-06%
Packets out: 8253
Anteil in: 0.99%
Packets in: 1379179843
Anteil null: 0.000020%
Packets null: 27831


#### Transport Protokoll

In [27]:
def split_transport(lf: pl.LazyFrame) -> tuple[pl.LazyFrame, ...]:
    protos = lf.select("ip.proto").unique().collect()["ip.proto"].to_list()
    return tuple(
        lf.filter(
            pl.col("ip.proto").is_null() if proto is None
            else pl.col("ip.proto") == proto
        )
        for proto in protos
    )

In [28]:
lf_by_proto = (
    lf.group_by("ip.proto")
    .agg(pl.len().alias("count"))
    .collect()
    .sort("count", descending=True)
)

In [29]:
total_packets: int  = lf.select(pl.len()).collect().item()
total_bytes: int = lf.select(pl.col("frame.len").sum()).collect().item()

t_start = lf.select(pl.col("frame.datetime").min()).collect().item()
t_end = lf.select(pl.col("frame.datetime").max()).collect().item()
if t_start and t_end != None:
    duration_s = (t_end - t_start).total_seconds()

print(f"Pakete gesamt : {total_packets:>12}")
print(f"Bytes gesamt  : {total_bytes:>12,}  ({total_bytes/1e6:.1f} MB)")
print(f"Zeitraum      : {t_start}  →  {t_end}")
print(f"Dauer         : {duration_s:.1f} s")
print(f"Ø Paketrate   : {total_packets/duration_s:>10,.1f} pkt/s")
print(f"Ø Bitrate     : {total_bytes*8/duration_s/1e6:>10,.1f} Mbit/s")

Pakete gesamt :   1392529795
Bytes gesamt  : 1,599,893,232,333  (1599893.2 MB)
Zeitraum      : 2025-08-13 15:51:49.573544  →  2025-08-13 19:39:55.926376
Dauer         : 13686.4 s
Ø Paketrate   :  101,745.9 pkt/s
Ø Bitrate     :      935.2 Mbit/s


#### Top-Talker (meiste Pakete / meiste Bytes)

In [30]:
talkers = (
    lf.group_by("ip.src")
    .agg([
        pl.len().alias("packets"),
        pl.col("frame.len").sum().alias("bytes"),
    ])
    .sort("bytes", descending=True)
)
top_talkers = talkers.head(20).collect()
top_talkers

ip.src,packets,bytes
str,u32,i64
"""163.181.207.4""",4061910,1957840620
"""163.181.207.2""",4053211,1953647702
"""163.181.207.1""",4051925,1953027850
"""163.181.207.3""",3876695,1868566990
"""195.69.189.18""",3766660,1815530120
…,…,…
"""201.62.49.58""",1531489,738177698
"""124.120.113.203""",1442965,695509130
"""217.73.144.39""",1379396,664868872


In [31]:
lf_ipv6 = lf.filter(
    pl.col("ipv6.src").is_not_null() | pl.col("ipv6.dst").is_not_null()
).collect()
lf_ipv6

frame.datetime,frame.len,frame.protocols,ip.src,ip.dst,ip.proto,ip.ttl,ipv6.src,ipv6.dst,ipv6.hlim,udp.srcport,udp.dstport,udp.length,tcp.srcport,tcp.dstport,tcp.len,dtls.record.content_type,dtls.record.version,dtls.handshake.type,coap.version,coap.type,coap.code,coap.mid,coap.token,coap.token_len,coap.opt.uri_path,coap.opt.uri_query,coap.opt.block_number,coap.opt.block_mflag,coap.opt.block_size,coap.opt.observe,coap.response_to,coap.response_in,coap.payload_length,source_file
datetime[μs],i64,str,str,str,str,str,str,str,str,str,str,i64,str,str,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
2025-08-13 15:51:51.835864,1374,"""eth:ethertype:ip:udp:teredo:ip…","""106.8.207.199""","""141.22.28.227""","""17""","""51""","""2f3e:3b74:6974:6c65:3d22:4765:…","""6c20:496e:666f:223b:6374:3d30:…","""60""","""5683""","""3544""",1340,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""data/raw/ddos_1/13082025_DDOS_…"
2025-08-13 15:51:52.796131,191,"""eth:ethertype:vlan:ethertype:i…",null,null,null,null,"""fe80::21a:e8ff:fe92:6664""","""ff02::1:2""","""1""","""546""","""547""",133,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""data/raw/ddos_1/13082025_DDOS_…"
2025-08-13 15:51:57.292536,191,"""eth:ethertype:vlan:ethertype:i…",null,null,null,null,"""fe80::21a:e8ff:fe99:e4cc""","""ff02::1:2""","""1""","""546""","""547""",133,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""data/raw/ddos_1/13082025_DDOS_…"
2025-08-13 15:51:57.478403,162,"""eth:ethertype:ip:udp:amt:ip:ip…","""181.199.119.134""","""141.22.28.227""","""17""","""118""","""3b49:6e73:7461:6e63:654e:616d:…","""5351:4c53:4552:5645:523b:4973:…","""82""","""1434""","""2268""",128,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""data/raw/ddos_1/13082025_DDOS_…"
2025-08-13 15:51:58.479987,191,"""eth:ethertype:vlan:ethertype:i…",null,null,null,null,"""fe80::21a:e8ff:fe97:c8f7""","""ff02::1:2""","""1""","""546""","""547""",133,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""data/raw/ddos_1/13082025_DDOS_…"
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
2025-08-13 19:39:43.326595,191,"""eth:ethertype:vlan:ethertype:i…",null,null,null,null,"""fe80::21a:e8ff:fe95:7fff""","""ff02::1:2""","""1""","""546""","""547""",133,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""data/raw/ddos_1/13082025_DDOS_…"
2025-08-13 19:39:44.179940,1374,"""eth:ethertype:ip:udp:teredo:ip…","""117.176.212.114""","""141.22.28.227""","""17""","""47""","""2f3e:3b74:6974:6c65:3d22:4765:…","""6c20:496e:666f:223b:6374:3d30:…","""60""","""5683""","""3544""",1340,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""data/raw/ddos_1/13082025_DDOS_…"
2025-08-13 19:39:48.173609,1506,"""eth:ethertype:ip:udp:mpls:ipv6…","""14.185.171.4""","""141.22.28.227""","""17""","""48""","""656e:636f:6469:6e67:3d22:5554:…","""3f3e:a3c:534f:4150:2d45:4e56:3…","""32""","""3702""","""6635""",3028,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""data/raw/ddos_1/13082025_DDOS_…"


In [32]:
lf_coap = lf.filter(
    pl.col("coap.version").is_not_null()
).collect()
lf_coap

frame.datetime,frame.len,frame.protocols,ip.src,ip.dst,ip.proto,ip.ttl,ipv6.src,ipv6.dst,ipv6.hlim,udp.srcport,udp.dstport,udp.length,tcp.srcport,tcp.dstport,tcp.len,dtls.record.content_type,dtls.record.version,dtls.handshake.type,coap.version,coap.type,coap.code,coap.mid,coap.token,coap.token_len,coap.opt.uri_path,coap.opt.uri_query,coap.opt.block_number,coap.opt.block_mflag,coap.opt.block_size,coap.opt.observe,coap.response_to,coap.response_in,coap.payload_length,source_file
datetime[μs],i64,str,str,str,str,str,str,str,str,str,str,i64,str,str,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
2025-08-13 15:51:49.573815,1374,"""eth:ethertype:ip:udp:coap""","""218.57.3.179""","""141.22.28.227""","""17""","""46""",null,null,null,"""5683""","""19077""",1340,null,null,null,null,null,null,"""1""","""2""","""69""","""32112""",null,"""0""",null,null,null,null,null,null,null,null,null,"""data/raw/ddos_1/13082025_DDOS_…"
2025-08-13 15:51:49.573971,1335,"""eth:ethertype:ip:udp:coap""","""183.141.175.241""","""141.22.28.227""","""17""","""51""",null,null,null,"""5683""","""28839""",1301,null,null,null,null,null,null,"""1""","""2""","""69""","""462""",null,"""0""",null,null,null,null,null,null,null,null,null,"""data/raw/ddos_1/13082025_DDOS_…"
2025-08-13 15:51:49.574021,1219,"""eth:ethertype:ip:udp:coap""","""111.50.9.66""","""141.22.28.227""","""17""","""48""",null,null,null,"""5683""","""53853""",1185,null,null,null,null,null,null,"""1""","""2""","""69""","""257""",null,"""0""",null,null,null,null,null,null,null,null,null,"""data/raw/ddos_1/13082025_DDOS_…"
2025-08-13 15:51:49.574340,1335,"""eth:ethertype:ip:udp:coap""","""111.38.220.83""","""141.22.28.227""","""17""","""49""",null,null,null,"""5683""","""25586""",1301,null,null,null,null,null,null,"""1""","""2""","""69""","""25968""",null,"""0""",null,null,null,null,null,null,null,null,null,"""data/raw/ddos_1/13082025_DDOS_…"
2025-08-13 15:51:49.574506,1506,"""eth:ethertype:ip:udp:coap""","""122.231.254.74""","""141.22.28.227""","""17""","""51""",null,null,null,"""5683""","""31621""",1524,null,null,null,null,null,null,"""1""","""2""","""69""","""25968""",null,"""0""",null,null,null,null,null,null,null,null,null,"""data/raw/ddos_1/13082025_DDOS_…"
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
2025-08-13 19:39:55.925889,1514,"""eth:ethertype:ip:udp:coap""","""115.45.28.90""","""141.22.28.227""","""17""","""42""",null,null,null,"""5683""","""47992""",1524,null,null,null,null,null,null,"""1""","""2""","""69""","""32112""",null,"""0""",null,null,null,null,null,null,null,null,null,"""data/raw/ddos_1/13082025_DDOS_…"
2025-08-13 19:39:55.926098,1335,"""eth:ethertype:ip:udp:coap""","""115.224.229.102""","""141.22.28.227""","""17""","""52""",null,null,null,"""5683""","""53231""",1301,null,null,null,null,null,null,"""1""","""2""","""69""","""462""",null,"""0""",null,null,null,null,null,null,null,null,null,"""data/raw/ddos_1/13082025_DDOS_…"
2025-08-13 19:39:55.926209,1490,"""eth:ethertype:ip:udp:coap""","""112.6.176.207""","""141.22.28.227""","""17""","""49""",null,null,null,"""5683""","""48149""",1524,null,null,null,null,null,null,"""1""","""2""","""69""","""257""",null,"""0""",null,null,null,null,null,null,null,null,null,"""data/raw/ddos_1/13082025_DDOS_…"


In [33]:
lf_dtls = lf.filter(
    pl.col("dtls.record.content_type").is_not_null() | pl.col("dtls.record.version").is_not_null() | pl.col("dtls.handshake.type").is_not_null()
).collect()
lf_dtls

frame.datetime,frame.len,frame.protocols,ip.src,ip.dst,ip.proto,ip.ttl,ipv6.src,ipv6.dst,ipv6.hlim,udp.srcport,udp.dstport,udp.length,tcp.srcport,tcp.dstport,tcp.len,dtls.record.content_type,dtls.record.version,dtls.handshake.type,coap.version,coap.type,coap.code,coap.mid,coap.token,coap.token_len,coap.opt.uri_path,coap.opt.uri_query,coap.opt.block_number,coap.opt.block_mflag,coap.opt.block_size,coap.opt.observe,coap.response_to,coap.response_in,coap.payload_length,source_file
datetime[μs],i64,str,str,str,str,str,str,str,str,str,str,i64,str,str,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
